Last Updated: 18/11/2026

In [0]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, accuracy_score, f1_score

from sklearn.svm import LinearSVC, SVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
 
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [0]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer

In [0]:
from joblib import Parallel, delayed, parallel_backend
from joblibspark import register_spark

In [0]:
RANDOM_STATE = 28082026

In [0]:
import ast
from sklearn.calibration import CalibratedClassifierCV

# Load the Datasets

In [0]:
df = pd.read_csv('/Workspace/Users/iamphillipluong@gmail.com/Datasets/exercise-classification/exercises_dataset_CLEAN.csv')

ex = pd.read_csv('/Volumes/projects/default/exercise-analysis/programs_detailed_boostcamp_kaggle_EDITED.csv')

# Exact-match bootstrap + Tfidf + SVM

In [0]:
def norm(s):
    s = str(s).lower()
    s = re.sub(r'\(.*?\)', '', s)          # strip equipment notes in parens
    s = re.sub(r'[^a-z0-9 ]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

In [0]:
labeled = df
labeled['norm'] = labeled['exercise'].apply(norm)

prog = ex
unique_names = pd.DataFrame({'exercise_name': prog['exercise_name'].unique()})
unique_names['norm'] = unique_names['exercise_name'].apply(norm)

In [0]:
# Layer 1: Exact-match bootstrap
lookup = labeled.drop_duplicates('norm').set_index('norm')['label']

unique_names['label'] = unique_names['norm'].map(lookup)
to_predict = unique_names[unique_names['label'].isna()].copy()
print(f"{unique_names['label'].notna().sum()} matched directly, {len(to_predict)} need prediction")

In [0]:
# Layer 2:Tfidf + SVM Classifier
vectorizer = FeatureUnion([
    ("word", TfidfVectorizer(ngram_range=(1,2), min_df=1, stop_words="english")),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5), min_df=1)),
])
pipe = Pipeline([("feats", vectorizer), ("clf", LinearSVC(class_weight="balanced", max_iter=5000))])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe, labeled['exercise'], labeled['label'], cv=cv, scoring='f1_macro', n_jobs=-1)
print("CV macro-F1:", scores.mean(), scores.std())

pipe.fit(labeled['exercise'], labeled['label'])
to_predict['label'] = pipe.predict(to_predict['exercise_name'])

## Probabilities to estimate errors

In [0]:
calibrated = CalibratedClassifierCV(pipe, cv=5, method='sigmoid')
calibrated.fit(labeled['exercise'], labeled['label'])

probs = calibrated.predict_proba(to_predict['exercise_name'])
to_predict['confidence'] = probs.max(axis=1)
to_predict.sort_values('confidence').head(30)  # review these first


# Random Forest Approach

In [0]:
# Layer 2:Tfidf + SVM Classifier
vectorizer = FeatureUnion([
    ("word", TfidfVectorizer(ngram_range=(1,2), min_df=1, stop_words="english")),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5), min_df=1)),
])

In [0]:
model = {"RandomForest": RandomForestClassifier(
            n_estimators=500,
            max_depth=None,
            min_samples_leaf=2,
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )}

In [0]:
from sklearn.pipeline import Pipeline

pipe = Pipeline([("feats", vectorizer), ("classifier", model['RandomForest'])])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe, labeled['exercise'], labeled['label'], cv=cv, scoring='f1_macro', n_jobs=-1)
print("CV macro-F1:", scores.mean(), scores.std())

pipe.fit(labeled['exercise'], labeled['label'])
to_predict['label'] = pipe.predict(to_predict['exercise_name'])

In [0]:
scores

# XGBoost and LightGBM

In [0]:
register_spark()

In [0]:
RANDOM_STATE = 4092026
N_SPLITS = 5
DATA_PATH = "/Workspace/Users/iamphillipluong@gmail.com/Datasets/exercise-classification/exercises_dataset_CLEAN.csv" 

# Load Datasets

In [0]:
def norm(s):
    s = str(s).lower()
    s = re.sub(r'\(.*?\)', '', s)          # strip equipment notes in parens
    s = re.sub(r'[^a-z0-9 ]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()
 
 
def load_data(path=DATA_PATH):
    df = pd.read_csv(path)
    df["norm_text"] = df["exercise"].apply(norm)
    return df
 

In [0]:
def make_vectorizer():
    return FeatureUnion([
        ("word", TfidfVectorizer(ngram_range=(1, 2), min_df=1, stop_words="english")),
        ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=1)),
    ])
 
 
def build_pipeline(clf):
    return Pipeline([("tfidf", make_vectorizer()), ("clf", clf)])

In [0]:
def build_models():
    return {
        "XGBoost": XGBClassifier(
            n_estimators=400,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            tree_method="hist",
            objective="multi:softprob",
            eval_metric="mlogloss",
            random_state=RANDOM_STATE,
        ),
        "LightGBM": LGBMClassifier(
            n_estimators=400,
            num_leaves=31,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            class_weight="balanced",
            objective="multiclass",
            random_state=RANDOM_STATE,
        ),
    }

In [0]:
def fit_and_score(model_name, train_idx, test_idx, X_text, y, sample_weight):
    clf = build_models()[model_name]
    pipe = build_pipeline(clf)
 
    X_train, X_test = X_text.iloc[train_idx], X_text.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
 
    fit_kwargs = {}
    if model_name == "XGBoost":
        fit_kwargs["clf__sample_weight"] = sample_weight[train_idx]
 
    pipe.fit(X_train, y_train, **fit_kwargs)
    preds = pipe.predict(X_test)
 
    return {
        "model": model_name,
        "f1_macro": f1_score(y_test, preds, average="macro"),
        "accuracy": accuracy_score(y_test, preds),
    }

In [0]:
def run_cv(df):
    le = LabelEncoder()
    y = le.fit_transform(df["label"])
    sample_weight = compute_sample_weight("balanced", y)
    X_text = df["norm_text"]
 
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    folds = list(skf.split(X_text, y))
 
    tasks = [
        (model_name, train_idx, test_idx)
        for model_name in build_models()
        for train_idx, test_idx in folds
    ]
 
    with parallel_backend("spark", n_jobs=len(tasks)):
        raw_results = Parallel(n_jobs=len(tasks))(
            delayed(fit_and_score)(model_name, train_idx, test_idx, X_text, y, sample_weight)
            for model_name, train_idx, test_idx in tasks
        )
 
    results_df = pd.DataFrame(raw_results)
    summary = (
        results_df.groupby("model")
        .agg(
            f1_macro_mean=("f1_macro", "mean"),
            f1_macro_std=("f1_macro", "std"),
            accuracy_mean=("accuracy", "mean"),
            accuracy_std=("accuracy", "std"),
        )
        .sort_values("f1_macro_mean", ascending=False)
    )
    return summary, results_df

In [0]:
def main():
    df = load_data()
    print("Label distribution:")
    print(df["label"].value_counts(), "\n")
 
    summary, per_fold = run_cv(df)
    print("per-fold results:")
    print(per_fold.to_string(index=False))
    print("\n=== 5-fold CV summary ===")
    print(summary.to_string())

In [0]:
df = load_data()
print("Label distribution:")
print(df["label"].value_counts(), "\n")

summary, per_fold = run_cv(df)
print("per-fold results:")
print(per_fold.to_string(index=False))
print("\n=== 5-fold CV summary ===")
print(summary.to_string())